# Материалы для научной статьи

Интерактивная замена `article_materials.py`. Ноутбук формирует компактные таблицы и версии рисунков из сохранённых результатов. Он не выполняет новые эксперименты.


In [ ]:
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('reports') if Path('reports').exists() else Path('../reports')
OUT = ROOT / 'article_materials'; TABLES = OUT / 'tables'; FIGURES = OUT / 'figures'
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)
metrics = pd.read_csv(ROOT / 'final_evaluation' / 'metrics.csv')
pairs = pd.read_csv(ROOT / 'error_analysis' / 'bootstrap_pairwise_differences.csv')


In [ ]:
article_metrics = metrics[['model','accuracy','precision_label_1','recall_label_1','f1_label_1','roc_auc','inference_seconds']].copy()
article_metrics.columns = ['Модель','Accuracy','Precision (AI=1)','Recall (AI=1)','F1 (AI=1)','ROC-AUC','Инференс CPU, с (24 628 текстов)']
article_metrics.to_csv(TABLES / 'table_3_final_metrics.csv', index=False, encoding='utf-8')
article_metrics.round(4)


In [ ]:
table_4 = []
for comparison, group in pairs.groupby('comparison', sort=False):
    f1 = group[group.metric == 'f1'].iloc[0]; auc = group[group.metric == 'roc_auc'].iloc[0]
    table_4.append({'Сравнение':comparison, 'ΔF1':f1.estimate, '95% CI ΔF1':f'[{f1.ci95_low:.4f}; {f1.ci95_high:.4f}]', 'ΔROC-AUC':auc.estimate, '95% CI ΔROC-AUC':f'[{auc.ci95_low:.4f}; {auc.ci95_high:.4f}]'})
bootstrap_table = pd.DataFrame(table_4); bootstrap_table.to_csv(TABLES / 'table_4_bootstrap.csv', index=False, encoding='utf-8'); bootstrap_table


In [ ]:
selected = {'figure_1_roc_curves': ROOT/'final_evaluation'/'roc_curves', 'figure_2_f1_by_length': ROOT/'error_analysis'/'f1_by_length', 'figure_4_fp_fn': ROOT/'error_analysis'/'fp_fn_comparison'}
for target, source in selected.items():
    for ext in ('.png','.pdf'): shutil.copy2(source.with_suffix(ext), FIGURES / f'{target}{ext}')

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharey=True)
for ax, metric, label in zip(axes, ['f1','roc_auc'], ['ΔF1','ΔROC-AUC']):
    view = pairs[pairs.metric == metric].reset_index(drop=True); y = range(len(view))
    ax.errorbar(view.estimate, y, xerr=[view.estimate-view.ci95_low, view.ci95_high-view.estimate], fmt='o', capsize=3)
    ax.axvline(0, color='gray', ls='--'); ax.set(xlabel=label, yticks=list(y), yticklabels=view.comparison); ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(FIGURES/'figure_3_bootstrap_differences.png', dpi=300); fig.savefig(FIGURES/'figure_3_bootstrap_differences.pdf'); plt.show()


## Рекомендуемая структура

1. Введение.  \n2. Набор данных и подготовка: характеристика CoAT и fixed split.  \n3. Модели и методика: TF-IDF, две MLP, LSTM, метрики и bootstrap.  \n4. Результаты и обсуждение: таблица итоговых метрик, ROC-кривые, анализ ошибок и длины, bootstrap.  \n5. Заключение и ограничения.

Полные текстовые заметки и проверенные источники находятся в `reports/article_materials/`.
